<a href="https://colab.research.google.com/github/liminalvoid/nlp/blob/main/sem_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Семинар 6. Токенайзеры, размеры тензеров и attention

## Установка и импорт библиотек, начальная настройка

In [ ]:
%pip install -q transformers sentencepiece

import random

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, logging


SEED = 52

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

logging.set_verbosity_error()

## Сравнение токенайзеров

В качестве дополнительного токенайзера использован токенайзер из модели [Qwen/Qwen2.5-7B](https://huggingface.co/Qwen/Qwen2.5-7B).

Число токенов зависит от словаря и алгоритма обучения. BPE-токенайзер строит словарь, начиная с отдельных символов и постепенно склеивая самые частотные пары. Какие пары склеятся — зависит от обучающего корпуса. Если в корпусе было много русского текста, частые слоги вроде «ого», «тион», «ский» станут отдельными токенами. Если русского было мало, слово будет разбито посимвольно. Отсюда разница: BERT (обучен в основном на английском) разобьёт «французских» на 5+ токенов, а Qwen (большой мультиязычный корпус) — на 1–2.

Код имеет совершенно другое распределение символов: много скобок, отступов, camelCase, snake_case, спецсимволов, поэтому он режется иначе. Современные токенайзеры (GPT-4, Qwen, LLaMA 3) специально включают код в обучающий корпус, поэтому у них в словаре есть целые паттерны вроде `def` , `    ` (4 пробела), `self`, `return`. Кроме того, многие токенайзеры применяют регулярные выражения перед токенизацией, которые по-разному разделяют буквы, цифры и пунктуацию — это напрямую влияет на то, где пройдут границы токенов в коде и в прозе.

BPE жадно склеивает частые пары. Если при обучении токенайзера русский текст составлял 0.1% корпуса, русские биграммы просто не наберут достаточно частоты, чтобы склеиться в длинные токены — и текст будет разбит почти посимвольно. В multilingual-моделях русского больше, поэтому появляются токены-слоги и даже целые слова. Результат: меньше токенов на то же предложение, а значит больше текста влезает в контекстное окно и инференс дешевле.

Byte-level BPE (GPT-2, LLaMA, Qwen) работает не с Unicode-символами, а с сырыми байтами. Кириллическая буква в UTF-8 — это 2 байта, китайский иероглиф — 3. Если пара байтов недостаточно частотна, чтобы склеиться, вы увидите в выводе отдельные байты, которые выглядят как мусор (как в случае с GPT-2 и Qwen): Ġ, Ð, µ и т.п. Это не ошибка — это просто отображение отдельных байтов UTF-8 через маппинг в печатные символы. По мере того как токенайзер обучается на большем количестве русского текста, эти байты склеиваются в нормальные слоги и слова.

In [ ]:
def inspect_tokenizer(tok_name: str, text: str, spacing_width: int = 16):
    tok = AutoTokenizer.from_pretrained(tok_name)
    encoded = tok(text, return_tensors="pt")
    tokens = tok.convert_ids_to_tokens(encoded["input_ids"][0])

    print(f"{'Text:':<{spacing_width}} {repr(text)}")
    print(f"{'Tokens:':<{spacing_width}} {str(tokens)}")
    print(f"{'Tokens count:':<{spacing_width}} {len(tokens)}")
    print(f"{'Vocab size:':<{spacing_width}} {tok.vocab_size}")
    print(f"{'Full vocab size:':<{spacing_width}} {len(tok.get_vocab())}\n")


texts = {
    "russian": "Съешь ещё этих мягких французских булок, да выпей чаю.",
    "code": "def print_str(string: str):\n    print(string)\n",
}
tokenizer_names = {
    "bert-base-uncased": "BERT WordPiece",
    "gpt2": "GPT-2 byte-level BPE",
    "google/mt5-small": "mT5 SentencePiece",
    "xlm-roberta-base": "XLM-R SentencePiece",
    "Qwen/Qwen2.5-7B": "Qwen SentencePiece",
}

for tok_name in tokenizer_names:
    print(f"=== {tok_name} ===")

    for (lang, text), i in zip(texts.items(), range(len(texts))):
        print(f"{i + 1}) {lang}")
        inspect_tokenizer(tok_name, text)

=== bert-base-uncased ===
1) russian
Text:            'Съешь ещё этих мягких французских булок, да выпей чаю.'
Tokens:          ['[CLS]', 'с', '##ъ', '##е', '##ш', '##ь', 'е', '##щ', '##е', 'э', '##т', '##и', '##х', 'м', '##я', '##г', '##к', '##и', '##х', 'ф', '##р', '##ан', '##ц', '##у', '##з', '##с', '##к', '##и', '##х', 'б', '##у', '##л', '##о', '##к', ',', 'д', '##а', 'в', '##ы', '##п', '##е', '##и', 'ч', '##а', '##ю', '.', '[SEP]']
Tokens count:    47
Vocab size:      30522
Full vocab size: 30522

2) code
Text:            'def print_str(string: str):\n    print(string)\n'
Tokens:          ['[CLS]', 'def', 'print', '_', 'st', '##r', '(', 'string', ':', 'st', '##r', ')', ':', 'print', '(', 'string', ')', '[SEP]']
Tokens count:    18
Vocab size:      30522
Full vocab size: 30522

=== gpt2 ===
1) russian
Text:            'Съешь ещё этих мягких французских булок, да выпей чаю.'
Tokens:          ['Ð', '¡', 'Ñ', 'Ĭ', 'Ðµ', 'Ñ', 'Ī', 'ÑĮ', 'ĠÐ', 'µ', 'Ñ', 'ī', 'Ñ', 'ĳ', 'Ġ', 'Ñ', 'į', 'ÑĤ

Ниже представлена табица числа токеном для разных текстов и токенайзеров.

In [ ]:
SPACING = 24

print(f"{'Tokenizer':<{SPACING}}", end="")

for lang in texts:
    print(f"┃ {lang:<{SPACING}}", end="")

print()

line = "━"
print(line * SPACING, end="", sep="")

for _ in range(len(texts)):
    print("╋", line * (SPACING + 1), end="", sep="")

print()

for tok_name, tok_description in tokenizer_names.items():
    tok = AutoTokenizer.from_pretrained(tok_name)
    print(f"{tok_name:<{SPACING}}", end="")

    for lang, text in texts.items():
        n = len(tok(text)["input_ids"])
        print(f"┃ {n:<{SPACING}}", end="")

    print()

Tokenizer               ┃ russian                 ┃ code                    
━━━━━━━━━━━━━━━━━━━━━━━━╋━━━━━━━━━━━━━━━━━━━━━━━━━╋━━━━━━━━━━━━━━━━━━━━━━━━━
bert-base-uncased       ┃ 47                      ┃ 18                      
gpt2                    ┃ 71                      ┃ 18                      
google/mt5-small        ┃ 22                      ┃ 15                      
xlm-roberta-base        ┃ 20                      ┃ 16                      
Qwen/Qwen2.5-7B         ┃ 23                      ┃ 11                      


## Расчет размеров

Дано: $B = 2$, $T = 7$, $d_{model} = 768$ и $h = 12$.

Найти: $Q$, $K$, $V$ и $QK^T$.

Решение:

$$
    d_k = \frac{d_{model}}{h} = \frac{768}{12} = 64
$$

$$
    Q, K, V: (B, h, T, d_k) \iff (2, 12, 7, 64)
$$

$$
    QK^T: (B, h, T, T) \iff (2, 12, 7, 7)
$$

## Поиск числа слоев, голов и размерности модели